In [1]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd
from abc import ABC, abstractmethod
import numpy as np
from sklearn.metrics import classification_report, r2_score, mean_absolute_error, root_mean_squared_error
from sklearn.preprocessing import LabelEncoder


In [2]:
class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, *, value=None):
        self.feature = feature       
        self.threshold = threshold   
        self.left = left             
        self.right = right          
        self.value = value           

    def is_leaf(self):
        return self.value is not None

Дерево для классификации

In [3]:
def gini_impurity(y):
    m = len(y)
    if m == 0:
        return 0
    counts = np.bincount(y)
    probabilities = counts / m
    return 1.0 - np.sum(probabilities ** 2)

def information_gain(y, left_y, right_y):
    p = len(left_y) / len(y)
    return gini_impurity(y) - p * gini_impurity(left_y) - (1 - p) * gini_impurity(right_y)


In [ ]:
class DecisionTreeClassifier:
    def __init__(self, max_depth=10, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None

    def fit(self, X, y):
        self.root = self._build_tree(X, y)
        return self

    def _build_tree(self, X, y, depth=0):
        n_samples, n_features = X.shape
        n_labels = len(np.unique(y))

        if (depth >= self.max_depth or 
            n_labels == 1 or 
            n_samples < self.min_samples_split):
            most_common = np.bincount(y).argmax()
            return Node(value=most_common)

        best_feat, best_thresh, best_gain = None, None, -1
        
        for feat_idx in range(n_features):
            X_column = X[:, feat_idx]
            thresholds = np.unique(X_column)
            
            for threshold in thresholds:
                left_idx = np.where(X_column <= threshold)[0]
                right_idx = np.where(X_column > threshold)[0]
                
                if len(left_idx) == 0 or len(right_idx) == 0:
                    continue
                    
                gain = information_gain(y, y[left_idx], y[right_idx])
                
                if gain > best_gain:
                    best_gain = gain
                    best_feat = feat_idx
                    best_thresh = threshold

        if best_gain <= 0:
            return Node(value=np.bincount(y).argmax())

        left_idx = np.where(X[:, best_feat] <= best_thresh)[0]
        right_idx = np.where(X[:, best_feat] > best_thresh)[0]
        
        left_child = self._build_tree(X[left_idx, :], y[left_idx], depth + 1)
        right_child = self._build_tree(X[right_idx, :], y[right_idx], depth + 1)
        
        return Node(feature=best_feat, threshold=best_thresh, left=left_child, right=right_child)

    def predict(self, X):
        return np.array([self._traverse_tree(x, self.root) for x in X])

    def _traverse_tree(self, x, node):
        if node.is_leaf():
            return node.value
            
        if x[node.feature] <= node.threshold:
            return self._traverse_tree(x, node.left)
        return self._traverse_tree(x, node.right)


In [10]:
data = pd.read_csv('../lessons/Dry_Bean_Dataset.csv', delimiter=';', decimal=',')
print(data.columns.tolist())
data = data.dropna(axis=1, how='all')  
data.fillna(data.mean(numeric_only=True), inplace=True)
X = data.drop(['Class'], axis=1).values
le = LabelEncoder()
y = le.fit_transform(data['Class'].values)
#y = data['Class'].values
#y_decoded = le.inverse_transform(y_encoded)  

print(f"Признаков: {X.shape[1]}, Объектов: {X.shape[0]}")

print(pd.Series(y).value_counts())

scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

['Area', 'Perimeter', 'MajorAxisLength', 'MinorAxisLength', 'AspectRation', 'Eccentricity', 'ConvexArea', 'EquivDiameter', 'Extent', 'Solidity', 'roundness', 'Compactness', 'ShapeFactor1', 'ShapeFactor2', 'ShapeFactor3', 'ShapeFactor4', 'Class']
Признаков: 16, Объектов: 13611
3    3546
6    2636
5    2027
4    1928
2    1630
0    1322
1     522
Name: count, dtype: int64


In [15]:
model = DecisionTreeClassifier( )
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

accuracy = classification_report(y_test, y_pred)
print(accuracy)

              precision    recall  f1-score   support

           0       0.92      0.77      0.84       261
           1       0.77      1.00      0.87       117
           2       0.87      0.88      0.87       317
           3       0.85      0.90      0.88       671
           4       0.91      0.94      0.92       408
           5       0.89      0.78      0.83       413
           6       0.81      0.81      0.81       536

    accuracy                           0.86      2723
   macro avg       0.86      0.87      0.86      2723
weighted avg       0.86      0.86      0.86      2723



Дерево для регрессии

In [12]:
def mse_impurity(y):
    m = len(y)
    if m == 0:
        return 0
    diff = y - np.mean(y)
    return np.sum(diff ** 2) / m

def information_gain(y, left_y, right_y):
    p = len(left_y) / len(y)
    return mse_impurity(y) - p * mse_impurity(left_y) - (1 - p) * mse_impurity(right_y)


In [13]:
class DecisionTreeRegration:
    def __init__(self, max_depth=10, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None

    def fit(self, X, y):
        self.root = self._build_tree(X, y)
        return self

    def _build_tree(self, X, y, depth=0):
        n_samples, n_features = X.shape
        n_labels = len(np.unique(y))

        if (depth >= self.max_depth or 
            n_labels == 1 or 
            n_samples < self.min_samples_split):
            most_common = np.mean(y)
            return Node(value=most_common)

        best_feat, best_thresh, best_gain = None, None, -1
        
        for feat_idx in range(n_features):
            X_column = X[:, feat_idx]
            thresholds = np.unique(X_column)
            
            for threshold in thresholds:
                left_idx = np.where(X_column <= threshold)[0]
                right_idx = np.where(X_column > threshold)[0]
                
                if len(left_idx) == 0 or len(right_idx) == 0:
                    continue
                    
                gain = information_gain(y, y[left_idx], y[right_idx])
                
                if gain > best_gain:
                    best_gain = gain
                    best_feat = feat_idx
                    best_thresh = threshold

        if best_gain <= 0:
            return Node(value=np.mean(y))

        left_idx = np.where(X[:, best_feat] <= best_thresh)[0]
        right_idx = np.where(X[:, best_feat] > best_thresh)[0]
        
        left_child = self._build_tree(X[left_idx, :], y[left_idx], depth + 1)
        right_child = self._build_tree(X[right_idx, :], y[right_idx], depth + 1)
        
        return Node(feature=best_feat, threshold=best_thresh, left=left_child, right=right_child)

    def predict(self, X):
        return np.array([self._traverse_tree(x, self.root) for x in X])

    def _traverse_tree(self, x, node):
        if node.is_leaf():
            return node.value
            
        if x[node.feature] <= node.threshold:
            return self._traverse_tree(x, node.left)
        return self._traverse_tree(x, node.right)


In [16]:
data = pd.read_csv('../lessons/hour.csv')
X = data.drop(['cnt','instant','dteday'], axis=1).values
y = data['cnt'].values

scaler_X = StandardScaler()
X = scaler_X.fit_transform(X)


scaler_y = StandardScaler()
y = scaler_y.fit_transform(y.reshape(-1, 1)).ravel()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Признаков: {X.shape[1]}, Объектов: {X.shape[0]}")




Признаков: 14, Объектов: 17379


In [17]:
model = DecisionTreeRegration( )
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

accuracy = r2_score(y_test, y_pred)
print(accuracy)

0.9986419015216151
